# 21. Spectrum Prediction in the Fractional Fourier Domain (SFFP)

Implements the **Spectral Fractional Filtering and Prediction (SFFP)** framework from:
**"Spectrum Prediction in the Fractional Fourier Domain With Adaptive Filtering"** (IEEE Wireless Communications Letters 2025).

## Paper (adapted to 72h→24h)
- **FrFT module:** Adaptive Fractional Fourier Transform (learnable order α) maps input into a fractional Fourier domain for better trend–noise separation.
- **Filter module:** Hybrid strategy (low-pass + random high fractional-frequency sampling) with learnable weights to suppress noise and retain predictive features.
- **Prediction module:** Complex-valued linear network in the FrFT domain; then **iFrFT** maps predictions back to time domain.
- **RevIN:** Reversible Instance Normalization for robustness to distribution shift.

## Same setup as 10–20
Data: work_dir/final, 72h→24h. Metrics: MAE, RMSE, MASE. Naive baseline. Same visuals.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
try:
    from scipy.linalg import dft
except ImportError:
    dft = None
print(f'TensorFlow: {tf.__version__}')


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_visible_devices(gpus, 'GPU')
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU enabled: {len(gpus)} device(s)')
        with tf.device('/GPU:0'): _ = tf.constant(1)
        print('GPU ready.')
    except RuntimeError as e: print('GPU config:', e)
else: print('No GPU. On Apple Silicon: pip install tensorflow-metal')
USE_GPU = len(gpus) > 0
num_cores = os.cpu_count() or 4
tf.config.threading.set_intra_op_parallelism_threads(num_cores)
tf.config.threading.set_inter_op_parallelism_threads(num_cores)


## Data loading (same as notebook 10)

In [ ]:
work_dir = Path("work_dir")
if not work_dir.exists():
    work_dir = Path("../work_dir")
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"
if not final_dir.exists() or not training_dir.exists() or not testing_dir.exists():
    raise FileNotFoundError("work_dir/final/training and testing not found")
class_options = sorted([d.name for d in training_dir.iterdir() if d.is_dir()])
LOOKBACK = 72
FORECAST_HORIZON = 24
print(f"Bands: {class_options}, Lookback={LOOKBACK}, Horizon={FORECAST_HORIZON}")

def load_data_for_band(band_name: str, split: str):
    split_dir = final_dir / split / band_name
    if not split_dir.exists(): return pd.DataFrame()
    dfs = []
    for p in sorted(split_dir.glob("final_*.parquet")):
        try: dfs.append(pd.read_parquet(p))
        except Exception as e: print(f"Error loading {p}: {e}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def prepare_next_day_sequences(train_df, test_df, lookback=72, forecast_horizon=24):
    freqs = sorted(train_df["freq_center_ghz"].unique())
    ths = sorted(train_df["threshold_dbm"].unique())
    if len(ths) > 1:
        train_df = train_df[train_df["threshold_dbm"] == ths[0]].copy()
        test_df = test_df[test_df["threshold_dbm"] == ths[0]].copy()
    X_tr, y_tr, X_te, y_te = [], [], [], []
    for freq in freqs:
        tr = train_df[train_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        te = test_df[test_df["freq_center_ghz"] == freq].sort_values(["date", "hour"])["au_pct"].values
        for i in range(len(tr) - lookback - forecast_horizon + 1):
            X_tr.append(tr[i:i+lookback])
            y_tr.append(tr[i+lookback:i+lookback+forecast_horizon])
        n_test_days = len(te) // forecast_horizon
        for d in range(n_test_days):
            if d == 0:
                inp = tr[-lookback:] if len(tr) >= lookback else np.concatenate([np.zeros(lookback - len(tr)), tr])
            else:
                h = max(0, lookback - d * forecast_horizon)
                inp = np.concatenate([tr[-h:], te[0:d*forecast_horizon]]) if h > 0 else te[d*forecast_horizon - lookback:d*forecast_horizon]
            tgt = te[d*forecast_horizon:(d+1)*forecast_horizon]
            if len(inp) == lookback and len(tgt) == forecast_horizon:
                X_te.append(inp)
                y_te.append(tgt)
    if not X_tr or not X_te:
        return np.array([]), np.array([]), np.array([]), np.array([])
    return np.array(X_tr).reshape(-1, lookback, 1), np.array(y_tr), np.array(X_te).reshape(-1, lookback, 1), np.array(y_te)


In [ ]:
train_data_by_band = {}
test_data_by_band = {}
for band in class_options:
    tr = load_data_for_band(band, "training")
    te = load_data_for_band(band, "testing")
    if not tr.empty and not te.empty:
        train_data_by_band[band] = tr
        test_data_by_band[band] = te
X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
for band in class_options:
    if band not in train_data_by_band: continue
    X_tr, y_tr, X_te, y_te = prepare_next_day_sequences(train_data_by_band[band], test_data_by_band[band], LOOKBACK, FORECAST_HORIZON)
    if len(X_tr) > 0 and len(X_te) > 0:
        X_train_list.append(X_tr)
        y_train_list.append(y_tr)
        X_test_list.append(X_te)
        y_test_list.append(y_te)
X_train = np.vstack(X_train_list)
y_train = np.vstack(y_train_list)
X_test = np.vstack(X_test_list)
y_test = np.vstack(y_test_list)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Discrete Fractional Fourier Transform (DFRFT) and SFFP model
DFRFT via eigendecomposition of DFT matrix: F^α = V @ D^α @ V^H. α is learnable (paper: adaptive FrFT).

In [ ]:
def build_dfrft_matrices(N):
    """Eigendecomposition of DFT matrix for DFRFT. Returns V (eigenvectors) and eigenvalue indices (0..3) for each."""
    if dft is not None:
        F = dft(N, scale='sqrtn')
    else:
        j = np.arange(N)
        k = j[:, None]
        F = np.exp(-2j * np.pi * j * k / N) / np.sqrt(N)
    eigvals, eigvecs = np.linalg.eig(F)
    eigvecs = np.asarray(eigvecs, dtype=np.complex128)
    idx = np.argsort(np.angle(eigvals))
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]
    k_group = np.round(2 * np.angle(eigvals) / np.pi).astype(int) % 4
    return eigvecs, k_group

def dfrft_matrix_for_alpha(V, k_group, alpha):
    """S_alpha = V @ D^alpha @ V^H. D^alpha has diagonal exp(-i pi alpha k/2) for k in k_group."""
    N = V.shape[0]
    lam_alpha = np.exp(-1j * np.pi * alpha * k_group / 2)
    D = np.diag(lam_alpha)
    return (V @ D @ V.conj().T).astype(np.complex64)

V72, k72 = build_dfrft_matrices(LOOKBACK)
V24, k24 = build_dfrft_matrices(FORECAST_HORIZON)
print('DFRFT eigenbasis computed for N=72 and N=24.')

In [ ]:
class RevIN(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    def build(self, input_shape):
        super().build(input_shape)
    def call(self, x, reverse=False):
        if reverse:
            return x * self.stored_std + self.stored_mean
        self.stored_mean = tf.reduce_mean(x, axis=-1, keepdims=True)
        self.stored_std = tf.math.reduce_std(x, axis=-1, keepdims=True) + 1e-5
        return (x - self.stored_mean) / self.stored_std

class FrFTLayer(layers.Layer):
    def __init__(self, V, k_group, inverse=False, **kwargs):
        super().__init__(**kwargs)
        self.V = tf.constant(V, dtype=tf.complex64)
        self.k_group = tf.constant(k_group, dtype=tf.float32)
        self.inverse = inverse
    def build(self, input_shape):
        self.alpha = self.add_weight('alpha', shape=(), initializer=keras.initializers.Constant(0.5), trainable=True)
        super().build(input_shape)
    def call(self, x):
        alpha = tf.cast(self.alpha, tf.float32)
        if self.inverse:
            alpha = -alpha
        lam = tf.exp(tf.complex(0., -np.pi * alpha * self.k_group / 2))
        D = tf.linalg.diag(lam)
        S = tf.matmul(self.V, tf.matmul(D, self.V, adjoint_b=True))
        x = tf.cast(x, tf.complex64)
        out = tf.matmul(x, S, adjoint_b=True)
        return out

class HybridFilterLayer(layers.Layer):
    def __init__(self, size, low_cutoff_ratio=0.5, **kwargs):
        super().__init__(**kwargs)
        self.size = size
        self.low_cutoff_ratio = low_cutoff_ratio
    def build(self, input_shape):
        self.W_real = self.add_weight('W_real', shape=(self.size,), initializer='ones', trainable=True)
        self.W_imag = self.add_weight('W_imag', shape=(self.size,), initializer='zeros', trainable=True)
        super().build(input_shape)
    def call(self, x_complex):
        re, im = tf.math.real(x_complex), tf.math.imag(x_complex)
        low_cut = int(self.size * self.low_cutoff_ratio)
        mask = np.ones(self.size, dtype=np.float32)
        mask[low_cut:] *= 0.3
        mask = tf.constant(mask)
        re = re * mask
        im = im * mask
        w = tf.complex(self.W_real, self.W_imag)
        return w * tf.complex(re, im)

class ComplexLinearLayer(layers.Layer):
    def __init__(self, out_dim, **kwargs):
        super().__init__(**kwargs)
        self.out_dim = out_dim
    def build(self, input_shape):
        in_dim = input_shape[-1]
        self.L_real = self.add_weight('L_real', shape=(in_dim, self.out_dim), initializer='glorot_uniform', trainable=True)
        self.L_imag = self.add_weight('L_imag', shape=(in_dim, self.out_dim), initializer='glorot_uniform', trainable=True)
        super().build(input_shape)
    def call(self, x_complex):
        re = tf.math.real(x_complex)
        im = tf.math.imag(x_complex)
        Re_o = tf.matmul(re, self.L_real) - tf.matmul(im, self.L_imag)
        Im_o = tf.matmul(im, self.L_real) + tf.matmul(re, self.L_imag)
        return tf.complex(Re_o, Im_o)


In [ ]:
def build_sffp_model():
    inp = layers.Input(shape=(LOOKBACK, 1))
    x = layers.Flatten()(inp)
    revin = RevIN()
    x_norm = revin(x)
    x_complex = tf.cast(x_norm, tf.complex64)
    frft = FrFTLayer(V72, k72, inverse=False)
    Xfr = frft(x_norm)
    filt = HybridFilterLayer(LOOKBACK, low_cutoff_ratio=0.5)
    Xf = filt(Xfr)
    pred_linear = ComplexLinearLayer(FORECAST_HORIZON)
    Xp = pred_linear(Xf)
    ifrft = FrFTLayer(V24, k24, inverse=True)
    pred_complex = ifrft(Xp)
    pred_time = tf.math.real(pred_complex)
    pred_time = revin(pred_time, reverse=True)
    return keras.Model(inp, pred_time)

model = build_sffp_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

## Train and evaluate

SFFP model (FrFT + Filter + Complex Linear + iFrFT + RevIN) is built above. Train on (X_train, y_train) and evaluate on test.


In [ ]:
# Model built above; training in next cell.

Train SFFP and compute metrics.

In [ ]:
BATCH = 128 if USE_GPU else 32
EPOCHS = 40
history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH, validation_split=0.2, verbose=1)
y_pred_sffp = model.predict(X_test, verbose=0)
y_pred_sffp = np.clip(y_pred_sffp, 0, 100).astype(np.float32)

def naive_predictor(X, horizon):
    last = X[:, -1, 0]
    return np.tile(last.reshape(-1, 1), (1, horizon))
y_pred_naive = naive_predictor(X_test, FORECAST_HORIZON)

def calculate_mae(y_true, y_pred):
    return mean_absolute_error(y_true.flatten(), y_pred.flatten())
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
def calculate_mase(y_true, y_pred, y_train):
    mae = np.mean(np.abs(y_true.flatten() - y_pred.flatten()))
    scale = np.mean(np.abs(np.diff(y_train.flatten()))) if len(y_train.flatten()) > 1 else 1.0
    return mae / max(scale, 1e-8)

mae_s = calculate_mae(y_test, y_pred_sffp)
rmse_s = calculate_rmse(y_test, y_pred_sffp)
mase_s = calculate_mase(y_test, y_pred_sffp, y_train)
mae_n = calculate_mae(y_test, y_pred_naive)
rmse_n = calculate_rmse(y_test, y_pred_naive)
mase_n = calculate_mase(y_test, y_pred_naive, y_train)

results_df = pd.DataFrame([
    {"Model": "SFFP (FrFT+Filter+Linear)", "MAE": mae_s, "RMSE": rmse_s, "MASE": mase_s},
    {"Model": "Naive Baseline", "MAE": mae_n, "RMSE": rmse_n, "MASE": mase_n},
])
results_df["MAE"] = results_df["MAE"].round(4)
results_df["RMSE"] = results_df["RMSE"].round(4)
results_df["MASE"] = results_df["MASE"].round(4)

print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Lookback: {LOOKBACK}h, Forecast: {FORECAST_HORIZON}h")
print(f"Test samples: {len(y_test)}")
print(results_df.to_string(index=False))
display(results_df)

## Analysis and Visualizations (same as notebook 10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MASE']
colors = ['#2ecc71', '#3498db', '#9b59b6']
for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric].values
    bars = ax.bar(results_df['Model'], vals, color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} by Model')
    ax.tick_params(axis='x', rotation=15)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02 * max(vals), f'{v:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.suptitle('Next-day prediction: SFFP (Fractional Fourier + Adaptive Filter)', y=1.02, fontsize=12)
plt.show()

naive_mae = results_df[results_df['Model'] == 'Naive Baseline']['MAE'].values[0]
naive_rmse = results_df[results_df['Model'] == 'Naive Baseline']['RMSE'].values[0]
naive_mase = results_df[results_df['Model'] == 'Naive Baseline']['MASE'].values[0]
improvement = results_df[results_df['Model'] != 'Naive Baseline'].copy()
improvement['MAE_imp_%'] = (1 - improvement['MAE'] / naive_mae) * 100
improvement['RMSE_imp_%'] = (1 - improvement['RMSE'] / naive_rmse) * 100
improvement['MASE_imp_%'] = (1 - improvement['MASE'] / naive_mase) * 100
print('Improvement over Naive Baseline (%):')
display(improvement[['Model', 'MAE_imp_%', 'RMSE_imp_%', 'MASE_imp_%']].round(2))
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(improvement))
w = 0.25
ax.bar(x - w, improvement['MAE_imp_%'], w, label='MAE', color='#2ecc71')
ax.bar(x, improvement['RMSE_imp_%'], w, label='RMSE', color='#3498db')
ax.bar(x + w, improvement['MASE_imp_%'], w, label='MASE', color='#9b59b6')
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(improvement['Model'], rotation=15)
ax.set_ylabel('Improvement (%)')
ax.legend()
ax.set_title('Improvement over Naive (positive = better)')
plt.tight_layout()
plt.show()

In [ ]:
hours = np.arange(FORECAST_HORIZON)
n_show = min(3, len(y_test))
fig, axes = plt.subplots(n_show, 1, figsize=(12, 4*n_show))
if n_show == 1: axes = [axes]
for i in range(n_show):
    ax = axes[i]
    ax.plot(hours, y_test[i], 'k--', linewidth=2.5, label='Actual', alpha=0.8)
    ax.plot(hours, y_pred_sffp[i], '-', linewidth=1.6, label='SFFP')
    ax.plot(hours, y_pred_naive[i], '-', linewidth=1.2, label='Naive', alpha=0.7)
    ax.set_title(f'Test sample {i+1}: Predicted vs Actual')
    ax.set_xlabel('Hour')
    ax.set_ylabel('AU (%)')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hours, y_test.mean(axis=0), 'k--', linewidth=2.5, label='Actual (mean)')
ax.plot(hours, y_pred_sffp.mean(axis=0), '-', linewidth=1.6, label='SFFP (mean)')
ax.plot(hours, y_pred_naive.mean(axis=0), '-', linewidth=1.2, label='Naive (mean)', alpha=0.7)
ax.set_title('Mean 24h profile: predicted vs actual')
ax.set_xlabel('Hour')
ax.set_ylabel('AU (%)')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

mae_per_hour = np.abs(y_test - y_pred_sffp).mean(axis=0)
mae_per_hour_n = np.abs(y_test - y_pred_naive).mean(axis=0)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours, mae_per_hour, '-o', label='SFFP', markersize=4)
ax.plot(hours, mae_per_hour_n, '-o', label='Naive', markersize=4, alpha=0.7)
ax.set_title('MAE by forecast hour')
ax.set_xlabel('Hour')
ax.set_ylabel('MAE (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
residuals_sffp = (y_test - y_pred_sffp).flatten()
residuals_naive = (y_test - y_pred_naive).flatten()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(residuals_sffp, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')
axes[0].set_title('Residuals: SFFP')
axes[1].hist(residuals_naive, bins=50, color='#95a5a6', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals: Naive')
plt.suptitle('Residual distribution', y=1.02)
plt.tight_layout()
plt.show()
print(f'Residual mean (bias): SFFP = {residuals_sffp.mean():.4f}, Naive = {residuals_naive.mean():.4f}')
print(f'Residual std:         SFFP = {residuals_sffp.std():.4f}, Naive = {residuals_naive.std():.4f}')

best_row = results_df[results_df['Model'] == 'SFFP (FrFT+Filter+Linear)'].iloc[0]
imp_mae = (1 - best_row['MAE'] / naive_mae) * 100
print(f"\nBest model: SFFP (MAE={best_row['MAE']:.4f}, RMSE={best_row['RMSE']:.4f}, MASE={best_row['MASE']:.4f}). Improvement over Naive: MAE {imp_mae:+.1f}%.")

### Key insights

- **Paper (IEEE WCL 2025):** SFFP uses adaptive FrFT (learnable α) for better trend–noise separation than time or standard Fourier domain; hybrid filtering in the fractional domain; complex-valued linear predictor; RevIN for distribution robustness.
- **Improvement over Naive:** Positive % means SFFP beats the last-value baseline.
- **Learned α:** After training, the FrFT layer’s α can be inspected (paper reports optimal α around 0.75–1.25 on RSS data).
